# Python Decorators: From Zero to Practical Use

You often see `@something` written **above** a function or method in Python.  
That line is called a **decorator**.

This notebook teaches decorators step by step:

1. What `@` really means
2. Built-in decorators you already meet in everyday code
3. How to write your own decorators
4. Real patterns used in production code

No external libraries required — only the Python standard library.

### Before you run any cell

Select this kernel in the notebook toolbar (top-right):

**`Python (langgraph-training)`**

That kernel uses this project's `.venv` where `ipykernel` is installed.  
If cells keep spinning with no output, the wrong Python interpreter is selected.


In [1]:
import sys

print("Kernel OK")
print(sys.executable)


Kernel OK
e:\Projects\langgraph-training\.venv\Scripts\python.exe


## Part 0 — Mental model: functions are objects

In Python, a function is a value like a string or a number. You can:

- Assign it to a variable
- Pass it to another function
- Return it from a function

A **decorator** is simply a function that takes a function, wraps it, and returns a new function.

### What `@decorator` means

This syntax:

```python
@decorator
def greet():
    print("Hello")
```

is **syntactic sugar** for:

```python
def greet():
    print("Hello")

greet = decorator(greet)
```

So the decorator runs **once** when Python defines the function, not every time you call it.

In [2]:
def shout(func):
    """A minimal decorator: wrap func and return a new callable."""

    def wrapper():
        print("--- before ---")
        func()
        print("--- after ---")

    return wrapper


def greet():
    print("Hello")


# Manual decoration (no @ syntax)
decorated_greet = shout(greet)
decorated_greet()

--- before ---
Hello
--- after ---


## Part 1 — Built-in decorators (beginner)

Python ships with several decorators. You do not write them — you **use** them.

| Decorator | Purpose |
|-----------|---------|
| `@property` | Expose a method as a read-only attribute |
| `@staticmethod` | Method that does not receive `self` or `cls` |
| `@classmethod` | Method that receives the class (`cls`) instead of an instance |

In [3]:
class BankAccount:
    def __init__(self, owner: str, balance: float):
        self.owner = owner
        self._balance = balance

    @property
    def balance(self) -> float:
        """Computed-style access: account.balance instead of account.balance()."""
        return self._balance

    @staticmethod
    def is_valid_amount(amount: float) -> bool:
        """Utility that does not need self or cls."""
        return amount > 0

    @classmethod
    def from_dict(cls, data: dict) -> "BankAccount":
        """Alternative constructor."""
        return cls(data["owner"], data["balance"])


account = BankAccount.from_dict({"owner": "Alex", "balance": 250.0})
print(account.owner, account.balance)
print(BankAccount.is_valid_amount(10))

Alex 250.0
True


## Part 2 — Your first custom decorator

A common pattern: run extra logic **before and after** the original function.

Here `log_calls` prints the function name and arguments, then calls the real function.

In [4]:
import functools


def log_calls(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        print(f"Calling {func.__name__} with args={args}, kwargs={kwargs}")
        result = func(*args, **kwargs)
        print(f"{func.__name__} returned {result!r}")
        return result

    return wrapper


@log_calls
def add(a: int, b: int) -> int:
    return a + b


add(2, 3)

Calling add with args=(2, 3), kwargs={}
add returned 5


5

## Part 3 — Preserve metadata with `functools.wraps`

Without care, the wrapped function **loses** its original name and docstring.

`@functools.wraps(func)` copies metadata from the original function to the wrapper.

In [5]:
import functools


def bad_decorator(func):
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)

    return wrapper


def good_decorator(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)

    return wrapper


@bad_decorator
def compute_tax(amount: float) -> float:
    """Calculate simple 10% tax."""
    return amount * 0.10


@good_decorator
def compute_discount(amount: float) -> float:
    """Calculate simple 5% discount."""
    return amount * 0.05


print("Without wraps:", compute_tax.__name__, compute_tax.__doc__)
print("With wraps:   ", compute_discount.__name__, compute_discount.__doc__)

Without wraps: wrapper None
With wraps:    compute_discount Calculate simple 5% discount.


## Part 4 — Decorators with arguments

Sometimes the decorator itself needs configuration, e.g. `@repeat(times=3)`.

Pattern (three nested functions):

1. **Outer** — receives decorator arguments (`times`)
2. **Middle** — receives the function being decorated
3. **Inner** — the actual wrapper called at runtime

In [ ]:
import time


def repeat(times: int):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            for _ in range(times):
                func(*args, **kwargs)

        return wrapper

    return decorator


def slow_down(seconds: float):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            time.sleep(seconds)
            return func(*args, **kwargs)

        return wrapper

    return decorator


@repeat(times=3)
def say_hi(name: str) -> None:
    print(f"Hi, {name}!")


say_hi("Sam")

## Part 5 — Stacking multiple decorators

You can apply more than one decorator to the same function.

```python
@timer
@log_calls
def work():
    ...
```

Python applies them **bottom-up**:

`work = timer(log_calls(work))`

So `log_calls` runs first during decoration; `timer` wraps the result.

In [ ]:
def timer(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - start
        print(f"[timer] {func.__name__} took {elapsed:.4f}s")
        return result

    return wrapper


@timer
@log_calls
def slow_add(a: int, b: int) -> int:
    time.sleep(0.2)
    return a + b


slow_add(10, 5)

## Part 6 — Practical patterns (intermediate)

These are patterns you will see in real projects.

In [ ]:
def retry(max_attempts: int = 3, delay_seconds: float = 0.1):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            last_error = None
            for attempt in range(1, max_attempts + 1):
                try:
                    return func(*args, **kwargs)
                except Exception as exc:  # noqa: BLE001 — demo only
                    last_error = exc
                    print(f"Attempt {attempt} failed: {exc}")
                    time.sleep(delay_seconds)
            raise last_error

        return wrapper

    return decorator


_call_count = 0


@retry(max_attempts=3, delay_seconds=0.05)
def flaky_service() -> str:
    global _call_count
    _call_count += 1
    if _call_count < 3:
        raise ConnectionError("temporary network error")
    return "success"


flaky_service()

In [ ]:
def require_role(required_role: str):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(user: dict, *args, **kwargs):
            if user.get("role") != required_role:
                raise PermissionError(
                    f"Role '{required_role}' required, got '{user.get('role')}'"
                )
            return func(user, *args, **kwargs)

        return wrapper

    return decorator


@require_role("admin")
def delete_user(user: dict, target_id: int) -> str:
    return f"User {target_id} deleted by {user['name']}"


admin = {"name": "Alex", "role": "admin"}
guest = {"name": "Sam", "role": "viewer"}

print(delete_user(admin, target_id=42))

try:
    delete_user(guest, target_id=42)
except PermissionError as exc:
    print("Blocked:", exc)

In [ ]:
from functools import lru_cache


@lru_cache(maxsize=None)
def fibonacci(n: int) -> int:
    if n < 2:
        return n
    return fibonacci(n - 1) + fibonacci(n - 2)


print("fib(30) =", fibonacci(30))
print("cache info:", fibonacci.cache_info())

## Part 7 — Decorating methods and classes (advanced)

### Methods

The same decorator pattern works on methods.  
Use `*args, **kwargs` so `self` is forwarded correctly.

### Class decorators

A decorator can also wrap a **class**.  
Below, `add_repr` injects a readable `__repr__` method.

In [ ]:
def trace_method(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        print(f"[trace] {func.__name__}")
        return func(*args, **kwargs)

    return wrapper


class Greeter:
    def __init__(self, title: str):
        self.title = title

    @trace_method
    def hello(self, name: str) -> str:
        return f"{self.title} {name}"


Greeter("Dr.").hello("Lee")

In [ ]:
def add_repr(cls):
    def __repr__(self):
        fields = ", ".join(f"{k}={v!r}" for k, v in self.__dict__.items())
        return f"{cls.__name__}({fields})"

    cls.__repr__ = __repr__
    return cls


@add_repr
class Point:
    def __init__(self, x: int, y: int):
        self.x = x
        self.y = y


Point(3, 4)

## Part 8 — Bridge to your LangGraph learning

Decorators appear throughout the Python ecosystem:

| Library | Example |
|---------|---------|
| **Pydantic** | `@field_validator` in `Lessons/02_Pydantic_Graph.ipynb` |
| **FastAPI** | `@app.get("/users")` registers a route |
| **pytest** | `@pytest.mark.slow` marks a test |
| **functools** | `@lru_cache` caches results |

LangGraph itself registers nodes with `graph.add_node("name", fn)` rather than `@`, but the same wrapping idea applies: you attach extra behavior to a callable without rewriting its core logic.

In [ ]:
# Minimal Pydantic-style validator using the same decorator idea

def field_validator(*field_names: str):
    def decorator(func):
        func._validated_fields = field_names
        return func

    return decorator


@field_validator("email")
def must_contain_at_sign(value: str) -> str:
    if "@" not in value:
        raise ValueError("email must contain @")
    return value


print(must_contain_at_sign._validated_fields)
print(must_contain_at_sign("user@example.com"))

## Part 9 — Recap cheat sheet

### When to use a decorator

- Cross-cutting behavior shared by many functions (logging, timing, auth, retries)
- Registering functions in a framework (routes, commands, validators)
- Wrapping without changing every call site

### Basic custom decorator

```python
import functools

def my_decorator(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        # before
        result = func(*args, **kwargs)
        # after
        return result
    return wrapper
```

### Decorator with arguments

```python
def my_decorator(arg):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            ...
        return wrapper
    return decorator
```

### Common pitfalls

1. Forgetting `@functools.wraps` — debugging tools show `wrapper` instead of your function name
2. Wrong stack order — remember: bottom decorator applied first
3. Heavy work inside the wrapper on every call — decoration-time setup belongs in the outer/middle functions

**Next in `Foundations/`:** context managers (`with`), generators, or typing deep-dive.